# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR^2 Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load and analyze the FAIR^2 clinical oncology dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/latest/) Python library, leveraging the Croissant data schema for reproducible and transparent data access.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Make sure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and tabular records from FAIR^2 using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)
# Retrieve metadata as a dict for convenience (but do not subscript or iterate Dataset.metadata directly)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review the available Record Sets, their fields, and the IDs that uniquely reference each data entity in FAIR^2.

A **RecordSet** in Croissant defines a named table (or collection of records), with each column defined by a **field**. All are referenced by unique `@id` strings. The following code lists all Record Sets and their fields by ID.

In [ ]:
# List all record sets by @id, and their field @id's. This provides the map for downstream analysis.
record_sets = [rs['@id'] for rs in dataset.metadata.record_sets]
print('Available record sets:')
for rs in dataset.metadata.record_sets:
    print(f"- Record Set @id: {rs['@id']} | name: {rs.get('name','(no name)')}")
    if 'fields' in rs:
        print("    Fields:")
        for field in rs['fields']:
            print(f"      - Field @id: {field['@id']} | name: {field.get('name','(no name)')} | dataType: {field.get('dataType','(unknown)')}")
    else:
        print("    (No fields listed)")

## 3. Data Extraction
Load all tables (record sets) as DataFrames, referencing them by their `@id`.

> Note: In FAIR^2, the main tabular record set is typically named or described in the schema with clinical, molecular, or demographic data columns. Find its `@id` above and use it for further steps.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {list(df.columns)}\n  Number of records: {len(df)}\n")

# For demo: pick the first available Record Set ID for exploration
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set for EDA: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())
else:
    print('No record sets available! Check dataset schema.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter, normalize, and group using numeric and categorical fields, referenced by their `@id` columns.

[NOTE: Edit the variable values below to match the `@id`s and column names of numeric and categorical fields you want to examine, as displayed in the previous cell.]

You might use age, diagnosis intervals, or similar numeric fields for demonstration.

In [ ]:
# ------ Customize the following @id's as per your chosen record set & fields ------

# Use main_record_set_id extracted in previous cell (could be something like 'cr:SecondPrimaryCRCRecords')
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Display sample column names for guidance
print('Available columns:', df.columns.tolist())

# Suppose age is stored under the field @id 'age' (replace with real @id as needed)
# For demonstration, select the first numeric-looking field as `numeric_field_id`
import numpy as np
numeric_field_id = None
for col in df.columns:
    # pick first column of type int or float, if possible
    if np.issubdtype(df[col].dropna().dtype, np.number):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # fallback: pick any column and try to convert to numeric (e.g. age column as string)
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='raise')
            numeric_field_id = col
            break
        except:
            continue

if numeric_field_id is None:
    raise RuntimeError('Could not identify a numeric field for demonstration.')

# Demonstration: filter by a threshold (e.g., age > 60)
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype != object else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize this numeric field
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, norm_col]].head())

# Group by a likely categorical field (pick the first non-numeric column)
group_field = None
for col in df.columns:
    if not np.issubdtype(df[col].dropna().dtype, np.number):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize field distributions or relationships using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Histogram of the chosen numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color="skyblue")
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.grid()
plt.show()

# If grouping variable is available, boxplot
if group_field:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated loading and exploring the FAIR^2 clinical oncology dataset using the `mlcroissant` library, referencing all entities and fields by their Croissant `@id` values for reproducibility. You can expand the analysis by adjusting field and record set variables to match your use case, performing advanced analysis as needed.